# Lahore groundwater forecasting: explained step by step

This notebook walks through the project's results. It **reads** the outputs of the pipeline
(`python scripts/run_all.py`) and reloads the saved models; it does not retrain them.

* **Question:** can we forecast depth to water at Lahore District piezometers for the *next observed
  season* (pre-monsoon or post-monsoon), and which inputs do the models rely on?
* **Data:** Punjab Irrigation Department / Directorate of Land Reclamation piezometer workbooks
  (2003 – pre-monsoon 2020) and NASA POWER daily weather.
* **Inspiration:** the WAFER project (World Aquifer Forecast Engine and Research,
  https://github.com/WAFERed). This is an independent implementation.

> Historical evaluation only. Nothing here is a verified current (2026) Lahore groundwater level,
> and ~30 monitoring points do not cover every neighbourhood.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from lahore_gw import plots
from lahore_gw.config import PROCESSED_DIR, REPORTS_DIR, MODELS_DIR
pd.set_option("display.width", 160)

## 1. Cleaning rules (what `scripts/01_clean_data.py` does)

1. Discover the files actually present in `data/raw/WL-LHR/` (4 Excel workbooks + a 2019 PDF).
2. Detect each header row (row index 1 for the 2003–2015 workbook) and recognise season columns
   such as `Pre-03`, `Po.09`, `Premonsoon 2014`.
3. Keep **District = Lahore** only, not the whole Lahore Irrigation Zone.
4. Verify that GIS number ↔ DLR number is one-to-one in every file, then use `LHR-<GIS>` as the ID.
5. `Closed`, `Dead` and blank cells are **missing readings**, never zero and never "aquifer dry".
6. Overlapping records (same well, year, season in two files) are compared and conflicts reported.
7. Coordinates are converted from degrees/minutes/seconds; depth to water (ft below ground) is kept
   separate from ground elevation (m above sea level).

In [ ]:
wells = pd.read_csv(PROCESSED_DIR / "lahore_wells.csv")
long = pd.read_csv(PROCESSED_DIR / "lahore_groundwater_long.csv")
print(len(wells), "monitoring locations;", len(long), "well-season cells")
long.status.value_counts()

In [ ]:
long[long.status == "reading"][["well_id", "year", "season", "depth_ft", "depth_m", "sources"]].head(8)

In [ ]:
pd.read_csv(REPORTS_DIR / "audit" / "coverage_by_season.csv").drop(columns="season_index")

Coverage falls over time: many piezometers are marked *Closed* (until 2016) or *Dead* (from 2017).
Those cells are gaps, not measurements.

### Coordinates and elevation
Some wells have different coordinates in the 2003–2015 file than in later files (up to ~28 km). The
latest file's coordinates are used and every discrepancy is listed, never silently corrected. Where files give different **ground elevations** (two wells), no file is assumed correct: the elevation is left unresolved, kept with its sources, and not used by the models:

In [ ]:
pd.read_csv(REPORTS_DIR / "audit" / "coordinate_discrepancies.csv")

In [ ]:
wells[["well_id", "village", "tehsil", "lat", "lon", "ground_elevation_m", "elevation_status", "coord_quality"]]

## 2. Weather and timing assumptions

The workbooks have no measurement dates. We assume the pre-monsoon survey reflects conditions at
the end of **May** and the post-monsoon survey at the end of **October**, and aggregate NASA POWER
daily data over:

* pre-monsoon window: 1 Nov (previous year) – 31 May
* post-monsoon window: 1 Jun – 31 Oct

NASA POWER is gridded at about 0.5° × 0.625°, so all Lahore wells share only a couple of
weather series. Weather therefore varies over *time*, not between neighbourhoods.

In [ ]:
wx = pd.read_csv(PROCESSED_DIR / "weather_seasonal_by_well.csv")
print(json.loads((PROCESSED_DIR / "weather_metadata.json").read_text())["distinct_weather_series"], "distinct series")
wx.groupby("season")[["rain_mm", "t2m_mean_c"]].describe().round(1).T

## 3. Forecasting set-up and leakage control

* One sample = reading at season *t* → reading at the next season *t+1* (only if both exist).
* Features are known at the end of season *t*: latest depth, last seasonal change, change vs the
  same season last year, target season type, rainfall/temperature in the window **ending** at *t*,
  and static location descriptors. Rainfall of the target window is *not* used (unknown at forecast time).
* Chronological split by target year: train ≤ 2012, validation 2013–2015, test 2016–2020.
* Imputation and scaling are fitted on training rows only; hyper-parameters are chosen on validation.

In [ ]:
from lahore_gw.features import build_samples, chronological_split, FEATURES
S = build_samples(long, wells, wx)
train, val, test = chronological_split(S, 2012, 2015)
pd.DataFrame({n: {"samples": len(d), "wells": d.well_id.nunique()} for n, d in
              [("train", train), ("validation", val), ("test", test)]})

## 4. Reload the saved models and reproduce the test errors

In [ ]:
import joblib
from lahore_gw.modeling import NNEnsemble, metrics, persistence
nn = NNEnsemble.load(MODELS_DIR / "nn")
ridge = joblib.load(MODELS_DIR / "ridge_model.joblib")
reproduced = pd.DataFrame([
    {"model": "Persistence", **metrics(test.target_depth_ft, persistence(test))},
    {"model": "Ridge", **metrics(test.target_depth_ft, ridge.predict(test))},
    {"model": "TensorFlow NN", **metrics(test.target_depth_ft, nn.predict(test))},
])
reproduced.round(3)

In [ ]:
met = pd.read_csv(REPORTS_DIR / "metrics.csv")
met.round(3)

In [ ]:
pd.read_csv(REPORTS_DIR / "metrics_vs_persistence.csv").round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
lim = [test.target_depth_ft.min() - 2, test.target_depth_ft.max() + 2]
ax.plot(lim, lim, "--", color=plots.NEUTRAL, lw=1, label="perfect")
ax.scatter(test.target_depth_ft, nn.predict(test), s=25, color=plots.SERIES[0], label="TensorFlow NN")
ax.set_xlabel("Observed depth (ft)"); ax.set_ylabel("Forecast depth (ft)"); ax.legend()
ax.set_title("Test period 2016–2020"); plt.show()

Levels look almost perfect because next season's depth is close to this season's depth. The
honest test is whether a model beats **persistence** (repeat the last reading) and the other simple
baselines. See the table above and `reports/results.md`.

## 5. Held-out feature importance

Permutation importance on the *test* period: shuffle one feature group and measure how much the
MAE grows. It shows a predictive association the fitted model relies on, **not** what causes groundwater change.
Inputs are correlated (rainfall and temperature share a coarse grid cell and season), so
reliance can shift between proxies.

In [ ]:
pd.read_csv(REPORTS_DIR / "feature_importance_nn.csv")[["feature_group", "mae_increase_ft", "std_ft"]].round(3)

In [ ]:
pd.read_csv(REPORTS_DIR / "feature_importance_ridge.csv")[["feature_group", "mae_increase_ft", "std_ft"]].round(3)

## 6. Forward forecast (clearly labelled)

In [ ]:
fwd_path = PROCESSED_DIR / "forward_forecast.csv"
pd.read_csv(fwd_path).round(2) if fwd_path.exists() else "no forward forecast" 

These are model outputs for the season after the last survey in the data (post-monsoon 2020).
No observation is available to verify them, and they are **not** current groundwater levels.

## 7. Limitations

* About 30 piezometers, many closed or dead in later years, so the test set is small and the
  wells that survive may not be representative.
* Measurement dates are unknown, so the weather windows rest on a seasonal assumption.
* NASA POWER weather is coarse (~50 km); there is no pumping, land-use, canal-flow or river data.
* Some coordinates disagree between source files (flagged in the audit).
* GRACE measures regional storage change over hundreds of km, not individual well depth, so it is
  kept as an optional, separate context script.